In [0]:
taxi_path="/Volumes/workspace/default/raw_data/taxi/archive.zip"

display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/taxi"))

In [0]:
import zipfile

zip_path="/Volumes/workspace/default/raw_data/taxi/archive.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files=z.namelist()
files

In [0]:
print('Databricks is working!')

In [0]:
import zipfile

zip_path="/Volumes/workspace/default/raw_data/taxi/archive.zip"
extract_path="/Volumes/workspace/default/raw_data/taxi/"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
    print('CSV extracted successfully!!')

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/taxi/"))

In [0]:
taxi_csv='/Volumes/workspace/default/raw_data/taxi/2018_Yellow_Taxi_Trip_Data.csv'

df=spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(taxi_csv)

print('Data loaded successfully')

In [0]:
df.count()

In [0]:
print('Rows :', df.count())
print('Columns :', len(df.columns))
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, to_timestamp

clean_df=(
    df.withColumn('pickup_datetime', to_timestamp(col('tpep_pickup_datetime'), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn('dropoff_datetime', to_timestamp(col('tpep_dropoff_datetime'), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn('trip_distance', to_timestamp(col('trip_distance').cast('double')))
    .withColumn('fare_amount', to_timestamp(col('fare_amount').cast('double')))
    .withColumn('tolls_amount', to_timestamp(col('tolls_amount').cast('double')))
    .withColumn('total_amount', to_timestamp(col('total_amount').cast('double')))
)

print('Cleaning transformation successfully!!')

In [0]:
display(
    clean_df.select(
        'pickup_datetime',
        'dropoff_datetime',
        'trip_distance',
        'fare_amount',
        'total_amount',
    ).limit(10)
)


In [0]:
clean_df.select(
        'pickup_datetime',
        'dropoff_datetime',
        'trip_distance',
        'fare_amount',
        'total_amount',
    ).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import col, to_timestamp, regexp_replace

clean_df=(
    df.withColumn('pickup_datetime', to_timestamp(col('tpep_pickup_datetime'), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn('dropoff_datetime', to_timestamp(col('tpep_dropoff_datetime'), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn('trip_distance', regexp_replace(col('trip_distance'),',','').cast('double'))
    .withColumn('fare_amount', regexp_replace(col('fare_amount'),',','').cast('double'))
    .withColumn('tolls_amount', regexp_replace(col('tolls_amount'),',','').cast('double'))
    .withColumn('total_amount', regexp_replace(col('total_amount'),',','').cast('double'))
)

print('Updated clean_df successfully!!')

In [0]:
from pyspark.sql.functions import col,sum

count_nulls=clean_df.select([sum(col(c).isNull().cast('int')).alias(c)
                             for c in clean_df.columns])

display(count_nulls)

In [0]:
Invalid_distance=clean_df.filter( col('trip_distance')<=0).count()

print('Invalid travel distance : ', Invalid_distance)

In [0]:
from pyspark.sql.functions import col, sum, regexp_replace

clean_df=clean_df.withColumn("fare_amount", regexp_replace(col("fare_amount"), ',', '').cast('double'))

display(clean_df.select("fare_amount").limit(10))

In [0]:
Invalid_fare=clean_df.filter( (col("fare_amount").isNull()) | (col("fare_amount")<=0 )).count()

print("Invalid Fare amount : ", Invalid_fare)

In [0]:
display(clean_df.limit(5))

In [0]:
Invalid_totalamount=clean_df.filter((col("total_amount").isNull()) | (col("total_amount")<=0)).count()

print("Invalid total amounts: ",Invalid_totalamount)

In [0]:
Invalid_passengers=clean_df.filter((col("passenger_count").isNull()) | (col("passenger_count")<=0)).count()

print("Invalid passengers: ", Invalid_passengers)

In [0]:
Invalid_datetime=clean_df.filter((col("pickup_datetime").isNull()) | (col("dropoff_datetime").isNull())).count()

print('Invalid date time: ', Invalid_datetime)

In [0]:
Invalid_triptime=clean_df.filter((col("pickup_datetime"))>=(col("dropoff_datetime"))).count()

print("Invalid trip time :",  Invalid_triptime)

In [0]:
from pyspark.sql.functions import col

final_df=clean_df.filter((col("passenger_count")>0)&
                         (col("trip_distance")>0)&
                         (col("fare_amount")>0)&
                         (col("total_amount")>0)&
                         (col("dropoff_datetime")>col("pickup_datetime")))

print("Rows :", clean_df.count())
print("Cleaned Rows :", final_df.count())

In [0]:
final_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.taxi_trip_cleaned")

In [0]:
spark.sql("""
          SELECT COUNT(*) AS total_rows FROM workspace.default.taxi_trip_cleaned
          """).show()

In [0]:
from pyspark.sql.functions import *
daily_revenue=spark.sql("""
                         SELECT
                         date(pickup_datetime) as trip_date,
                            COUNT(*) AS total_trips,
                            ROUND(SUM(total_amount),2) AS total_revenue,
                            ROUND(AVG(total_amount),2) AS avg_trip_amount,
                            ROUND(AVG(trip_distance),2) AS avg_trip_distance
                            FROM workspace.default.taxi_trip_cleaned
                         GROUP BY DATE(pickup_datetime)
                         ORDER BY trip_date
                         """)
display(daily_revenue)

In [0]:
daily_revenue.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.daily_revenue")

In [0]:
from pyspark.sql.functions import *
monthly_revenue=spark.sql("""
                          SELECT YEAR(pickup_datetime) AS year,
                          MONTH(pickup_datetime) AS month,
                          COUNT(*) AS total_trips,
                          ROUND(SUM(total_amount),2) AS total_revenue,
                          ROUND(AVG(total_amount),2) AS avg_trip_amount,
                          ROUND(AVG(trip_distance),2) AS avg_trip_distance
                          FROM 
                          workspace.default.taxi_trip_cleaned
                          GROUP BY
                          YEAR(pickup_datetime),
                          MONTH(pickup_datetime)
                          ORDER BY
                          year, month
                          """)
display(monthly_revenue)

In [0]:
monthly_revenue.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.monthly_revenue")

In [0]:
payment_analysis=spark.sql("""
                           SELECT payment_type,
                           COUNT(*) AS total_trips,
                           ROUND(SUM(total_amount),2) AS total_revenue,
                           ROUND(AVG(total_amount),2) AS avg_trip_amount
                           FROM
                           workspace.default.taxi_trip_cleaned
                           GROUP BY payment_type
                           ORDER BY total_trips DESC
                           """)
display(payment_analysis)

In [0]:
payment_analysis.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.payment_analysis")

In [0]:
vendor_analysis=spark.sql("""
                          SELECT VendorID,
                          COUNT(*) AS total_trips,
                          ROUND(SUM(total_amount),2) AS total_revenue,
                          ROUND(AVG(total_amount),2) AS avg_trip_amount,
                          ROUND(AVG(trip_distance),2) AS avg_trip_distance
                          FROM
                          workspace.default.taxi_trip_cleaned
                          GROUP BY VendorID
                          ORDER BY total_trips DESC
                          """)
display(vendor_analysis)

In [0]:
vendor_analysis.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.vendor_analysis")